# Pandas Sorting Lesson — Student Dataset (100 rows)
1. Build a dataset with messy city names
2. See WHY messy data breaks sorting
3. Fix every misspelling
4. Learn sorting: `sort_values`, `sort_index`, `nlargest` / `nsmallest`
5. Exercises + solutions

In [ ]:
import pandas as pd
import numpy as np

pd.set_option("display.width", 120)        # wider output so tables don't wrap
pd.set_option("display.max_columns", 20)   # show all columns

np.random.seed(42)  # same "random" data every run, so all students see identical results

N = 100

first_names = ["Aziz", "Malika", "Bekzod", "Dilnoza", "Jasur", "Nilufar", "Sardor", "Kamola",
               "Otabek", "Madina", "Javohir", "Sevara", "Shahzod", "Gulnoza", "Ulugbek", "Zarina",
               "Rustam", "Feruza", "Doniyor", "Shahnoza", "Timur", "Laylo", "Bobur", "Munisa", "Akmal"]

clean_cities = ["Tashkent", "Samarkand", "Bukhara", "Namangan", "Andijan"]

df = pd.DataFrame({
    # name    -> student's first name + student number (so every name is unique)
    "name": [f"{np.random.choice(first_names)}_{i+1:03d}" for i in range(N)],
    # city    -> where the student lives (some values will be MISSPELLED on purpose below)
    "city": np.random.choice(clean_cities, size=N, p=[0.35, 0.25, 0.2, 0.1, 0.1]),
    # age     -> student's age in years (18–25)
    "age": np.random.randint(18, 26, size=N),
    # hours   -> hours studied per week (0–10)
    "hours": np.random.randint(0, 11, size=N),
    # math    -> math exam score (40–100); NaN = student missed the exam
    "math": np.random.randint(40, 101, size=N).astype(float),
    # english -> English exam score (40–100); NaN = student missed the exam
    "english": np.random.randint(40, 101, size=N).astype(float),
})

# Add some missing exam scores (NaN)
df.loc[np.random.choice(N, 6, replace=False), "math"] = np.nan
df.loc[np.random.choice(N, 6, replace=False), "english"] = np.nan

print("Shape (rows, columns):", df.shape)

Shape (rows, columns): (100, 6)


## Inject realistic mistakes into `city`
These are the kinds of errors you get from manual data entry / Google Forms.

In [ ]:
typos = {
    3:  "tashkent",      # all lowercase
    11: "TASHKENT",      # all uppercase
    17: "Toshkent",      # Uzbek spelling
    24: " Tashkent",     # leading space (invisible!)
    29: "Tashkent ",     # trailing space (invisible!)
    35: "Tashkant",      # typo
    41: "Samarqand",     # Uzbek spelling
    46: "Samarkant",     # typo
    52: "samarkand",     # lowercase
    58: "Bukhoro",       # Uzbek spelling
    63: "Buhara",        # missing letter
    69: "BUKHARA ",      # uppercase + trailing space
    74: "Namangn",       # missing letter
    80: "Andijon",       # Uzbek spelling
    87: "andijan",       # lowercase
}
for idx, bad in typos.items():
    df.loc[idx, "city"] = bad

print("=== First 10 rows ===")
print(df.head(10))

=== First 10 rows ===
           name       city  age  hours  math  english
0    Sardor_001   Tashkent   20      7  47.0     58.0
1  Shahnoza_002  Samarkand   21      6  92.0     97.0
2   Ulugbek_003  Samarkand   21      2  99.0     71.0
3   Javohir_004   tashkent   20      0  83.0      NaN
4    Kamola_005   Tashkent   21      0  83.0     95.0
5     Timur_006   Tashkent   20     10  44.0     44.0
6    Sardor_007    Bukhara   19      2  78.0     84.0
7   Doniyor_008    Bukhara   20      5  43.0     43.0
8     Bobur_009    Bukhara   24      6  45.0     55.0
9   Javohir_010    Andijan   20      5  84.0     63.0


## Step 1 — Spot the problem
There should be only 5 cities, but `value_counts()` shows many more.

In [ ]:
print("Unique city values:", df["city"].nunique())   # expected 5, we get many more
print()
print("=== City counts (BEFORE fix) ===")
print(df["city"].value_counts())

Unique city values: 20

=== City counts (BEFORE fix) ===
city
Tashkent     31
Samarkand    25
Bukhara      15
Andijan       7
Namangan      7
tashkent      1
TASHKENT      1
Toshkent      1
 Tashkent     1
Tashkent      1
Tashkant      1
Samarqand     1
Samarkant     1
samarkand     1
Bukhoro       1
Buhara        1
BUKHARA       1
Namangn       1
Andijon       1
andijan       1
Name: count, dtype: int64


In [ ]:
# Spaces are invisible in normal output — repr() shows them inside quotes
print("=== Every city value with quotes (to see hidden spaces) ===")
for city in sorted(df["city"].unique()):
    print(repr(city))

=== Every city value with quotes (to see hidden spaces) ===
' Tashkent'
'Andijan'
'Andijon'
'BUKHARA '
'Buhara'
'Bukhara'
'Bukhoro'
'Namangan'
'Namangn'
'Samarkand'
'Samarkant'
'Samarqand'
'TASHKENT'
'Tashkant'
'Tashkent'
'Tashkent '
'Toshkent'
'andijan'
'samarkand'
'tashkent'


## Step 2 — Why this breaks sorting
Python sorts strings by character code: **space < UPPERCASE < lowercase**.
So `" Tashkent"` comes first, and `"andijan"` / `"tashkent"` drop to the very end.

In [ ]:
print("=== Sorted by city — TOP 5 (messy data) ===")
print(df.sort_values("city")[["name", "city"]].head(5))
print()
print("=== Sorted by city — BOTTOM 5 (messy data) ===")
print(df.sort_values("city")[["name", "city"]].tail(5))

=== Sorted by city — TOP 5 (messy data) ===
           name       city
24    Timur_025   Tashkent
76   Malika_077    Andijan
78  Dilnoza_079    Andijan
93   Munisa_094    Andijan
22  Nilufar_023    Andijan

=== Sorted by city — BOTTOM 5 (messy data) ===
           name       city
29    Akmal_030  Tashkent 
17    Laylo_018   Toshkent
87    Timur_088    andijan
52   Otabek_053  samarkand
3   Javohir_004   tashkent


## Step 3 — Fix the city names
Each fix handles a different type of mistake.

In [ ]:
# Fix A: remove leading/trailing spaces   " Tashkent" / "Tashkent " / "BUKHARA "  -> no spaces
df["city"] = df["city"].str.strip()

# Fix B: unify capitalization   "tashkent" / "TASHKENT" / "samarkand" / "andijan" -> "Title Case"
df["city"] = df["city"].str.title()

print("=== City counts after strip() + title() ===")
print(df["city"].value_counts())   # remaining problems are real spelling mistakes

=== City counts after strip() + title() ===
city
Tashkent     35
Samarkand    26
Bukhara      16
Andijan       8
Namangan      7
Toshkent      1
Tashkant      1
Samarqand     1
Samarkant     1
Bukhoro       1
Buhara        1
Namangn       1
Andijon       1
Name: count, dtype: int64


In [ ]:
# Fix C: map every remaining misspelling to the correct name
city_fix = {
    "Toshkent":  "Tashkent",    # Uzbek spelling
    "Tashkant":  "Tashkent",    # typo
    "Samarqand": "Samarkand",   # Uzbek spelling
    "Samarkant": "Samarkand",   # typo
    "Bukhoro":   "Bukhara",     # Uzbek spelling
    "Buhara":    "Bukhara",     # missing letter
    "Namangn":   "Namangan",    # missing letter
    "Andijon":   "Andijan",     # Uzbek spelling
}
df["city"] = df["city"].replace(city_fix)

print("Unique city values:", df["city"].nunique())   # now exactly 5
print()
print("=== City counts (AFTER fix) ===")
print(df["city"].value_counts())

Unique city values: 5

=== City counts (AFTER fix) ===
city
Tashkent     37
Samarkand    28
Bukhara      18
Andijan       9
Namangan      8
Name: count, dtype: int64


In [ ]:
# Safety check — any city NOT in our clean list? Should be 0.
bad_rows = df[~df["city"].isin(clean_cities)]
print("Rows with a wrong city:", len(bad_rows))

Rows with a wrong city: 0


## Step 4 — Sorting
### 4.1 Sort by one column (ascending is the default)

In [ ]:
print("=== Sorted by age (youngest first) ===")
print(df.sort_values("age").head())

=== Sorted by age (youngest first) ===
          name       city  age  hours  math  english
15  Munisa_016  Samarkand   18     10  94.0     71.0
14  Kamola_015   Tashkent   18      7   NaN     67.0
24   Timur_025   Tashkent   18      2  58.0     72.0
21  Sevara_022  Samarkand   18      4  99.0     74.0
25    Aziz_026   Tashkent   18      0  56.0    100.0


### 4.2 Descending order

In [ ]:
print("=== Sorted by math (highest first) ===")
print(df.sort_values("math", ascending=False).head())

=== Sorted by math (highest first) ===
           name       city  age  hours   math  english
54   Malika_055    Andijan   20     10  100.0     65.0
2   Ulugbek_003  Samarkand   21      2   99.0     71.0
21   Sevara_022  Samarkand   18      4   99.0     74.0
29    Akmal_030   Tashkent   23      2   97.0     51.0
41    Jasur_042  Samarkand   19      2   97.0     85.0


### 4.3 Where do NaN values go?
By default NaN goes to the **end**. Use `na_position="first"` to see who missed the exam.

In [ ]:
print("=== Sorted by math, NaN first (who missed the exam) ===")
print(df.sort_values("math", na_position="first").head(8))

=== Sorted by math, NaN first (who missed the exam) ===
           name       city  age  hours  math  english
14   Kamola_015   Tashkent   18      7   NaN     67.0
35  Doniyor_036   Tashkent   19      2   NaN     82.0
47   Feruza_048  Samarkand   22      2   NaN     62.0
63   Rustam_064    Bukhara   25      5   NaN     84.0
80   Zarina_081    Andijan   19      6   NaN     87.0
96   Otabek_097    Bukhara   25      5   NaN     74.0
42  Doniyor_043    Bukhara   18      1  40.0     41.0
52   Otabek_053  Samarkand   18      6  40.0     65.0


### 4.4 Sort by multiple columns
First by city (A→Z), then inside each city by math (high→low).

In [ ]:
print("=== Sorted by city A→Z, then math high→low ===")
print(df.sort_values(["city", "math"], ascending=[True, False]).head(10))

=== Sorted by city A→Z, then math high→low ===
           name     city  age  hours   math  english
54   Malika_055  Andijan   20     10  100.0     65.0
40   Bekzod_041  Andijan   18      0   97.0     70.0
76   Malika_077  Andijan   18      3   91.0      NaN
9   Javohir_010  Andijan   20      5   84.0     63.0
22  Nilufar_023  Andijan   20      2   81.0      NaN
87    Timur_088  Andijan   25      9   69.0     62.0
78  Dilnoza_079  Andijan   18      7   51.0     88.0
93   Munisa_094  Andijan   18      3   44.0     52.0
80   Zarina_081  Andijan   19      6    NaN     87.0
99     Aziz_100  Bukhara   25      0   97.0     56.0


### 4.5 `sort_values` returns a NEW DataFrame
The original `df` is unchanged unless you reassign it (or use `inplace=True`).

In [ ]:
sorted_df = df.sort_values("hours")
print("Original index (first 5):", df.index[:5].tolist())         # original order
print("Sorted index   (first 5):", sorted_df.index[:5].tolist())  # index labels moved with the rows

Original index (first 5): [0, 1, 2, 3, 4]
Sorted index   (first 5): [3, 4, 20, 19, 25]


### 4.6 Reset the index after sorting

In [ ]:
print("=== Sorted by hours (high→low) with a fresh 0,1,2... index ===")
print(df.sort_values("hours", ascending=False).reset_index(drop=True).head())

=== Sorted by hours (high→low) with a fresh 0,1,2... index ===
          name       city  age  hours   math  english
0   Munisa_016  Samarkand   18     10   94.0     71.0
1   Bekzod_017    Bukhara   25     10   79.0     66.0
2    Timur_006   Tashkent   20     10   44.0     44.0
3  Gulnoza_063  Samarkand   18     10   65.0     89.0
4   Malika_055    Andijan   20     10  100.0     65.0


### 4.7 Sort by the index — `sort_index()` restores original order

In [ ]:
print("=== sorted_df put back in original order with sort_index() ===")
print(sorted_df.sort_index().head())

=== sorted_df put back in original order with sort_index() ===
           name       city  age  hours  math  english
0    Sardor_001   Tashkent   20      7  47.0     58.0
1  Shahnoza_002  Samarkand   21      6  92.0     97.0
2   Ulugbek_003  Samarkand   21      2  99.0     71.0
3   Javohir_004   Tashkent   20      0  83.0      NaN
4    Kamola_005   Tashkent   21      0  83.0     95.0


### 4.8 Sort with a custom key — e.g. by length of the city name

In [ ]:
print("=== Cities sorted by name length (shortest first) ===")
print(df.sort_values("city", key=lambda col: col.str.len())[["name", "city"]].drop_duplicates("city"))

=== Cities sorted by name length (shortest first) ===
           name       city
6    Sardor_007    Bukhara
9   Javohir_010    Andijan
27    Laylo_028   Tashkent
23   Malika_024   Namangan
62  Gulnoza_063  Samarkand


### 4.9 Shortcuts: `nlargest` / `nsmallest`

In [ ]:
print("=== Top 5 English scores (nlargest) ===")
print(df.nlargest(5, "english"))
print()
print("=== 5 students with fewest study hours (nsmallest) ===")
print(df.nsmallest(5, "hours"))

=== Top 5 English scores (nlargest) ===
            name       city  age  hours  math  english
25      Aziz_026   Tashkent   18      0  56.0    100.0
43    Sardor_044   Namangan   25      8  86.0    100.0
70     Laylo_071   Tashkent   25      8  67.0    100.0
1   Shahnoza_002  Samarkand   21      6  92.0     97.0
60   Ulugbek_061   Tashkent   23      0  42.0     97.0

=== 5 students with fewest study hours (nsmallest) ===
           name       city  age  hours  math  english
3   Javohir_004   Tashkent   20      0  83.0      NaN
4    Kamola_005   Tashkent   21      0  83.0     95.0
19   Malika_020   Tashkent   19      0  52.0     51.0
20   Munisa_021  Samarkand   25      0  89.0     89.0
25     Aziz_026   Tashkent   18      0  56.0    100.0


## Step 5 — Exercises
1. Show the 10 youngest students.
2. Sort by English score high→low, put students who missed the exam at the top.
3. Sort by city A→Z, then by age youngest→oldest.
4. Create a `total` column (math + english) and show the top 5 students.
5. Find the student with the most study hours in **each city**.
6. Sort cities by their average math score (high→low).

## Solutions

In [ ]:
# 1. 10 youngest students
print("=== Exercise 1: 10 youngest students ===")
print(df.sort_values("age").head(10))

=== Exercise 1: 10 youngest students ===
           name       city  age  hours  math  english
15   Munisa_016  Samarkand   18     10  94.0     71.0
14   Kamola_015   Tashkent   18      7   NaN     67.0
24    Timur_025   Tashkent   18      2  58.0     72.0
21   Sevara_022  Samarkand   18      4  99.0     74.0
25     Aziz_026   Tashkent   18      0  56.0    100.0
42  Doniyor_043    Bukhara   18      1  40.0     41.0
40   Bekzod_041    Andijan   18      0  97.0     70.0
37    Bobur_038    Bukhara   18      3  51.0     68.0
51   Feruza_052   Tashkent   18      5  92.0     68.0
52   Otabek_053  Samarkand   18      6  40.0     65.0


In [ ]:
# 2. English high→low, NaN first
print("=== Exercise 2: English high→low, missed exam first ===")
print(df.sort_values("english", ascending=False, na_position="first").head(10))

=== Exercise 2: English high→low, missed exam first ===
           name       city  age  hours  math  english
3   Javohir_004   Tashkent   20      0  83.0      NaN
22  Nilufar_023    Andijan   20      2  81.0      NaN
39    Akmal_040   Tashkent   21      6  40.0      NaN
58   Sevara_059    Bukhara   21      0  68.0      NaN
61   Bekzod_062  Samarkand   25      8  51.0      NaN
76   Malika_077    Andijan   18      3  91.0      NaN
25     Aziz_026   Tashkent   18      0  56.0    100.0
43   Sardor_044   Namangan   25      8  86.0    100.0
70    Laylo_071   Tashkent   25      8  67.0    100.0
60  Ulugbek_061   Tashkent   23      0  42.0     97.0


In [ ]:
# 3. City A→Z, then age ascending
print("=== Exercise 3: City A→Z, then age ===")
print(df.sort_values(["city", "age"]).head(10))

=== Exercise 3: City A→Z, then age ===
           name     city  age  hours   math  english
40   Bekzod_041  Andijan   18      0   97.0     70.0
76   Malika_077  Andijan   18      3   91.0      NaN
78  Dilnoza_079  Andijan   18      7   51.0     88.0
93   Munisa_094  Andijan   18      3   44.0     52.0
80   Zarina_081  Andijan   19      6    NaN     87.0
9   Javohir_010  Andijan   20      5   84.0     63.0
22  Nilufar_023  Andijan   20      2   81.0      NaN
54   Malika_055  Andijan   20     10  100.0     65.0
87    Timur_088  Andijan   25      9   69.0     62.0
37    Bobur_038  Bukhara   18      3   51.0     68.0


In [ ]:
# 4. Total score, top 5  (NaN + number = NaN, so missing-exam students fall to the bottom)
df["total"] = df["math"] + df["english"]
print("=== Exercise 4: Top 5 by total score ===")
print(df.sort_values("total", ascending=False).head(5))

=== Exercise 4: Top 5 by total score ===
            name       city  age  hours  math  english  total
1   Shahnoza_002  Samarkand   21      6  92.0     97.0  189.0
92     Timur_093  Samarkand   21      0  93.0     94.0  187.0
43    Sardor_044   Namangan   25      8  86.0    100.0  186.0
11    Munisa_012   Tashkent   25      5  91.0     94.0  185.0
26    Sevara_027   Tashkent   19      0  95.0     90.0  185.0


In [ ]:
# 5. Most study hours in each city: sort, then keep the first row per city
print("=== Exercise 5: Most study hours in each city ===")
print(df.sort_values("hours", ascending=False).drop_duplicates("city").sort_values("city"))

=== Exercise 5: Most study hours in each city ===
          name       city  age  hours   math  english  total
54  Malika_055    Andijan   20     10  100.0     65.0  165.0
16  Bekzod_017    Bukhara   25     10   79.0     66.0  145.0
43  Sardor_044   Namangan   25      8   86.0    100.0  186.0
15  Munisa_016  Samarkand   18     10   94.0     71.0  165.0
5    Timur_006   Tashkent   20     10   44.0     44.0   88.0


In [ ]:
# 6. Cities by average math score
print("=== Exercise 6: Cities by average math score ===")
print(df.groupby("city")["math"].mean().sort_values(ascending=False).round(1))

=== Exercise 6: Cities by average math score ===
city
Andijan      77.1
Samarkand    76.4
Namangan     71.9
Tashkent     70.5
Bukhara      65.2
Name: math, dtype: float64
